# A3 — Classificador com CNN Pré-treinada: EfficientNet-B0

**Projeto:** EcoVision AI  
**Dataset:** `pavansanagapati/images-dataset` (Kaggle)  
**Arquitetura:** EfficientNet-B0 pré-treinada no ImageNet

## Objetivos

Nesta atividade vamos:

1. preparar e validar o dataset;
2. remover duplicatas exatas antes do split;
3. construir divisão estratificada de treino/validação/teste;
4. aplicar data augmentation;
5. carregar uma CNN pré-treinada;
6. substituir o classification head para 7 classes;
7. executar **feature extraction** com backbone congelado;
8. executar **fine-tuning** do modelo completo;
9. documentar curvas de loss e accuracy;
10. reportar accuracy global e por classe;
11. discutir quando usar feature extraction ou fine-tuning;
12. justificar a EfficientNet-B0 e as estratégias de augmentation.

## Requisitos de execução

- **Ambiente recomendado:** Google Colab com GPU T4
- **GPU:** T4 (16 GB de VRAM)
- **RAM recomendada:** 8 GB ou mais
- **Espaço em disco recomendado:** 2–3 GB
- **Imagem de entrada:** 224 × 224
- **Batch size:** 32
- **Classes:** 7
- **Seed:** 42
- **Tempo estimado com checkpoint existente:** ~2–5 minutos
- **Tempo estimado treinando no Colab T4:** aproximadamente 5–15 minutos, dependendo do ambiente

## 1. Preparação do ambiente

A EfficientNet-B0 será utilizada por **transfer learning**.

Transfer learning significa começar com parâmetros que já aprenderam representações visuais em um grande dataset, em vez de inicializar todos os pesos aleatoriamente.

Neste caso:

```text
ImageNet
   ↓
EfficientNet-B0 pré-treinada
   ↓
troca do classification head
   ↓
7 classes do nosso dataset
```

In [ ]:
import sys
import subprocess

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "kagglehub",
        "scikit-learn",
        "matplotlib",
        "pandas",
        "pillow",
    ],
    check=True,
)

print("Dependências verificadas.")

In [ ]:
from pathlib import Path
import os
import sys

def encontrar_raiz_projeto():
    candidatos = []

    cwd = Path.cwd().resolve()
    candidatos.extend([cwd, *cwd.parents])

    candidatos.extend([
        Path("/content/EcoVision-AI-organized"),
        Path("/content/EcoVision-AI"),
    ])

    for candidato in candidatos:
        if (
            (candidato / "src" / "cnn").exists()
            and (candidato / "src" / "common").exists()
        ):
            return candidato

    try:
        from google.colab import drive

        drive.mount("/content/drive")

        mydrive = Path("/content/drive/MyDrive")

        for nome in ["EcoVision-AI-organized", "EcoVision-AI"]:
            for candidato in mydrive.glob(f"**/{nome}"):
                if (
                    (candidato / "src" / "cnn").exists()
                    and (candidato / "src" / "common").exists()
                ):
                    return candidato.resolve()

    except Exception:
        pass

    raise FileNotFoundError(
        "Não encontrei a raiz do EcoVision-AI. "
        "No Colab, coloque a pasta do projeto em /content "
        "ou no Google Drive."
    )

PROJECT_ROOT = encontrar_raiz_projeto()

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Raiz do projeto:", PROJECT_ROOT)
print("Diretório atual:", Path.cwd())

In [ ]:
import torch

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("PyTorch:", torch.__version__)
print("Dispositivo:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"VRAM: {total_vram:.2f} GB")
else:
    print(
        "O notebook também funciona em CPU, "
        "mas o fine-tuning será mais lento."
    )

## 2. Dataset e problema de duplicatas

O dataset possui sete categorias:

```text
bike
cars
cats
dogs
flowers
horses
human
```

Durante a inspeção inicial encontramos uma estrutura aninhada que fazia uma busca recursiva contar **3.606 arquivos**, embora o conjunto útil contivesse aproximadamente metade disso.

Depois de escolher a raiz correta das sete classes, foram identificadas **1.803 imagens válidas**.

Antes do treinamento também calculamos SHA-256 para detectar arquivos exatamente iguais. Foram removidas **39 duplicatas exatas**, resultando em:

```text
1.803 imagens válidas
-   39 duplicatas
-------------------
1.764 imagens únicas
```

Essa etapa é importante porque uma imagem duplicada em treino e teste causa **data leakage**: o modelo pode ser avaliado em uma imagem que já viu durante o treinamento.

### Preparação portátil para Colab

O script histórico `clean_dataset.py` foi criado inicialmente no Windows e contém o caminho local usado no desenvolvimento. Para que este notebook seja reprodutível em Colab, a célula abaixo prepara o dataset de forma **portátil** a partir do caminho retornado pelo KaggleHub.

O processo é equivalente:

```text
download
 ↓
localizar pasta que contém as 7 classes
 ↓
validar imagens com PIL
 ↓
copiar dataset limpo
 ↓
SHA-256 global
 ↓
remover duplicatas exatas
 ↓
data/cnn_dedup
```

In [ ]:
from collections import defaultdict
from PIL import Image
import hashlib
import shutil
import kagglehub

EXPECTED_CLASSES = {
    "bike",
    "cars",
    "cats",
    "dogs",
    "flowers",
    "horses",
    "human",
}

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}

CNN_CLEAN_DIR = PROJECT_ROOT / "data" / "cnn_clean"
CNN_DEDUP_DIR = PROJECT_ROOT / "data" / "cnn_dedup"


def contar_imagens_diretas(pasta):
    return sum(
        1
        for arquivo in pasta.iterdir()
        if arquivo.is_file()
        and arquivo.suffix.lower() in IMAGE_EXTENSIONS
    )


def localizar_raiz_classes(download_root):
    candidatos = [Path(download_root)]
    candidatos.extend(
        p
        for p in Path(download_root).rglob("*")
        if p.is_dir()
    )

    encontrados = []

    for candidato in candidatos:
        try:
            filhos = {
                p.name: p
                for p in candidato.iterdir()
                if p.is_dir()
            }
        except PermissionError:
            continue

        if EXPECTED_CLASSES.issubset(filhos.keys()):
            total = sum(
                contar_imagens_diretas(filhos[classe])
                for classe in EXPECTED_CLASSES
            )
            encontrados.append((total, candidato))

    if not encontrados:
        raise FileNotFoundError(
            "Não encontrei uma pasta com as sete classes esperadas."
        )

    encontrados.sort(
        key=lambda item: item[0],
        reverse=True,
    )

    return encontrados[0][1]


def imagem_valida(caminho):
    try:
        with Image.open(caminho) as imagem:
            imagem.verify()
        return True
    except Exception:
        return False


def sha256_arquivo(caminho):
    sha = hashlib.sha256()

    with open(caminho, "rb") as arquivo:
        for bloco in iter(lambda: arquivo.read(8192), b""):
            sha.update(bloco)

    return sha.hexdigest()


def preparar_dataset_cnn():
    if CNN_DEDUP_DIR.exists():
        classes_existentes = {
            p.name
            for p in CNN_DEDUP_DIR.iterdir()
            if p.is_dir()
        }

        total_existente = sum(
            contar_imagens_diretas(p)
            for p in CNN_DEDUP_DIR.iterdir()
            if p.is_dir()
        )

        if (
            EXPECTED_CLASSES.issubset(classes_existentes)
            and total_existente > 0
        ):
            print("Dataset deduplicado já existe.")
            print("Imagens:", total_existente)
            return CNN_DEDUP_DIR

    print("Baixando/localizando dataset pelo KaggleHub...")

    download_root = Path(
        kagglehub.dataset_download(
            "pavansanagapati/images-dataset"
        )
    )

    source_dir = localizar_raiz_classes(
        download_root
    )

    print("Raiz das classes:", source_dir)

    if CNN_CLEAN_DIR.exists():
        shutil.rmtree(CNN_CLEAN_DIR)

    CNN_CLEAN_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    validas = 0
    invalidas = 0

    for classe in sorted(EXPECTED_CLASSES):
        origem = source_dir / classe
        destino = CNN_CLEAN_DIR / classe

        destino.mkdir(
            parents=True,
            exist_ok=True,
        )

        for arquivo in sorted(origem.iterdir()):
            if (
                not arquivo.is_file()
                or arquivo.suffix.lower() not in IMAGE_EXTENSIONS
            ):
                continue

            if not imagem_valida(arquivo):
                invalidas += 1
                continue

            shutil.copy2(
                arquivo,
                destino / arquivo.name,
            )

            validas += 1

    print("Imagens válidas:", validas)
    print("Imagens inválidas:", invalidas)

    grupos = defaultdict(list)

    for classe_dir in CNN_CLEAN_DIR.iterdir():
        if not classe_dir.is_dir():
            continue

        for arquivo in classe_dir.iterdir():
            if (
                arquivo.is_file()
                and arquivo.suffix.lower() in IMAGE_EXTENSIONS
            ):
                grupos[sha256_arquivo(arquivo)].append(
                    arquivo
                )

    if CNN_DEDUP_DIR.exists():
        shutil.rmtree(CNN_DEDUP_DIR)

    CNN_DEDUP_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    copiadas = 0
    duplicatas_removidas = 0
    conflitos_classes = 0

    for arquivos in grupos.values():
        classes = {
            arquivo.parent.name
            for arquivo in arquivos
        }

        if len(classes) == 1:
            escolhido = arquivos[0]
            classe = escolhido.parent.name

            destino_classe = CNN_DEDUP_DIR / classe
            destino_classe.mkdir(
                parents=True,
                exist_ok=True,
            )

            shutil.copy2(
                escolhido,
                destino_classe / escolhido.name,
            )

            copiadas += 1
            duplicatas_removidas += len(arquivos) - 1

        else:
            conflitos_classes += 1

    for classe in EXPECTED_CLASSES:
        (CNN_DEDUP_DIR / classe).mkdir(
            parents=True,
            exist_ok=True,
        )

    print()
    print("Hashes únicos:", len(grupos))
    print("Imagens copiadas:", copiadas)
    print("Duplicatas removidas:", duplicatas_removidas)
    print("Conflitos entre classes:", conflitos_classes)
    print("Dataset final:", CNN_DEDUP_DIR)

    return CNN_DEDUP_DIR


dataset_path = preparar_dataset_cnn()

In [ ]:
contagem_classes = {}

for classe in sorted(EXPECTED_CLASSES):
    classe_dir = dataset_path / classe

    contagem_classes[classe] = sum(
        1
        for arquivo in classe_dir.iterdir()
        if arquivo.is_file()
        and arquivo.suffix.lower() in IMAGE_EXTENSIONS
    )

print("Distribuição final:")

for classe, quantidade in contagem_classes.items():
    print(f"{classe:12s}: {quantidade}")

print()
print("Total:", sum(contagem_classes.values()))

## 3. Split estratificado

Depois da deduplicação, o dataset é dividido em:

```text
70% treino
15% validação
15% teste
```

O parâmetro `stratify` preserva aproximadamente a proporção de cada classe nos três conjuntos.

No experimento validado:

```text
Treino:     1.234
Validação:    265
Teste:        265
```

A deduplicação ocorre **antes** do split. Depois disso, também verificamos se nenhum hash aparece em conjuntos diferentes.

In [ ]:
from src.cnn.data_pipeline import (
    criar_datasets,
    criar_dataloaders,
)

dataset_base, train_dataset, val_dataset, test_dataset = criar_datasets(
    dataset_path
)

train_loader, val_loader, test_loader = criar_dataloaders(
    train_dataset,
    val_dataset,
    test_dataset,
)

print("Classes:", dataset_base.class_to_idx)
print()
print("Total:", len(dataset_base))
print("Treino:", len(train_dataset))
print("Validação:", len(val_dataset))
print("Teste:", len(test_dataset))

In [ ]:
subprocess.run(
    [sys.executable, "-m", "src.cnn.check_duplicates"],
    cwd=PROJECT_ROOT,
    check=True,
)

## 4. Data augmentation

O pipeline de treino utiliza quatro estratégias.

### 4.1 RandomResizedCrop

```python
RandomResizedCrop(
    224,
    scale=(0.8, 1.0)
)
```

Cria variações de enquadramento e escala.

**Benefício:** o modelo aprende que o objeto pode aparecer em posições/tamanhos ligeiramente diferentes.

**Risco:** crops agressivos poderiam remover partes fundamentais de uma bicicleta, carro, animal ou pessoa. Por isso o limite mínimo foi mantido em `0.8`.

### 4.2 RandomHorizontalFlip

```python
RandomHorizontalFlip(p=0.5)
```

É adequado para as sete classes porque inverter esquerda/direita não muda a categoria.

Entretanto, seria inadequado para tarefas em que orientação, texto ou lateralidade fossem semanticamente relevantes.

### 4.3 RandomRotation

```python
RandomRotation(degrees=10)
```

Pequenas rotações simulam variações naturais da câmera.

Uma rotação muito agressiva poderia produzir carros, bicicletas, pessoas ou animais em orientações improváveis.

### 4.4 ColorJitter

```python
ColorJitter(
    brightness=0.15,
    contrast=0.15,
    saturation=0.10
)
```

Ajuda o modelo a tolerar variações de iluminação e câmera.

O ajuste é propositalmente leve. Alterações intensas de cor poderiam prejudicar principalmente `flowers`, em que cor pode conter informação discriminativa, e também modificar pistas de pelagem em animais.

## 5. Normalização do ImageNet

A EfficientNet-B0 foi pré-treinada com imagens normalizadas segundo as estatísticas do ImageNet.

Por isso usamos:

```text
mean = (0.485, 0.456, 0.406)
std  = (0.229, 0.224, 0.225)
```

Para cada canal:

\[
x_{norm} = rac{x-\mu}{\sigma}
\]

Usar a mesma normalização mantém a distribuição das entradas compatível com aquela vista durante o pré-treinamento.

In [ ]:
from src.cnn.data_pipeline import (
    criar_transforms,
    IMAGENET_MEAN,
    IMAGENET_STD,
)

train_transform, eval_transform = criar_transforms()

print("Transformações de treino:")
print(train_transform)

print()
print("Transformações de avaliação:")
print(eval_transform)

print()
print("ImageNet mean:", IMAGENET_MEAN)
print("ImageNet std:", IMAGENET_STD)

## 6. Por que EfficientNet-B0?

A EfficientNet-B0 foi escolhida por oferecer um bom equilíbrio entre:

- capacidade de representação;
- número relativamente reduzido de parâmetros;
- velocidade;
- memória;
- disponibilidade de pesos ImageNet;
- compatibilidade confortável com Colab T4.

Nosso dataset possui apenas **7 classes** e **1.764 imagens únicas**. Um backbone muito maior aumentaria o custo e poderia ampliar o risco de overfitting sem garantia de benefício.

Além disso, as classes são objetos naturais comuns — veículos, animais, flores e pessoas — relativamente próximos do tipo de conteúdo visual presente no ImageNet. Portanto, as features pré-treinadas são um ponto de partida adequado.

## 7. Arquitetura adaptada

A EfficientNet-B0 original possui um classification head para **1.000 classes ImageNet**.

No EcoVision AI:

```text
EfficientNet-B0
      ↓
backbone convolucional
      ↓
features 1280-D
      ↓
Dropout
      ↓
Linear(1280, 7)
      ↓
logits das 7 classes
```

O novo `Linear` possui:

\[
1280 	imes 7 + 7 = 8.967
\]

parâmetros treináveis durante a fase inicial de feature extraction.

In [ ]:
import inspect

from src.cnn.models.pretrained_cnn import (
    EcoVisionPretrainedCNN,
    contar_parametros,
)

print(inspect.getsource(EcoVisionPretrainedCNN))

In [ ]:
modelo = EcoVisionPretrainedCNN(
    num_classes=7,
    pretrained=True,
)

entrada = torch.randn(
    2,
    3,
    224,
    224,
)

logits = modelo(
    entrada
)

print("Entrada:", entrada.shape)
print("Logits:", logits.shape)

total, treinaveis = contar_parametros(
    modelo
)

print(f"Parâmetros totais: {total:,}")
print(f"Treináveis antes do freeze: {treinaveis:,}")

modelo.congelar_backbone()

total, treinaveis_frozen = contar_parametros(
    modelo
)

print()
print(f"Treináveis com backbone congelado: {treinaveis_frozen:,}")

assert logits.shape == (2, 7)
assert treinaveis_frozen == 8967

print("Modelo adaptado: OK")

## 8. Feature Extraction × Fine-Tuning

### Feature Extraction

Primeiro congelamos o backbone:

```text
features convolucionais → congeladas
classification head     → treinável
```

Apenas os **8.967 parâmetros do novo head** são atualizados.

Isso é útil quando:

- o dataset novo é pequeno;
- o domínio é relativamente próximo do pré-treinamento;
- queremos reduzir risco de overfitting;
- queremos uma adaptação inicial rápida.

### Fine-Tuning

Depois liberamos todo o modelo:

```text
backbone + head → treináveis
```

Agora os filtros convolucionais também podem se adaptar ao dataset específico.

### Estratégia adotada

Como temos um dataset pequeno e classes visualmente próximas do ImageNet, utilizamos:

```text
FASE 1
Feature Extraction
2 epochs
LR = 1e-3

        ↓

FASE 2
Fine-Tuning completo
até 8 epochs
LR = 1e-4
```

O learning rate menor no fine-tuning reduz a chance de alterar agressivamente as features já aprendidas.

## 9. Treinamento

Configuração:

| Parâmetro | Valor |
|---|---:|
| Loss | CrossEntropyLoss |
| Optimizer | AdamW |
| Head epochs | 2 |
| Head learning rate | 1e-3 |
| Fine-tuning epochs | até 8 |
| Fine-tuning learning rate | 1e-4 |
| Weight decay | 0.01 |
| Early stopping patience | 3 |
| Batch size | 32 |

O melhor checkpoint é escolhido pela menor **validation loss**.

In [ ]:
CNN_CHECKPOINT = (
    PROJECT_ROOT
    / "models"
    / "ecovision_cnn_pretrained_best.pth"
)

if CNN_CHECKPOINT.exists():
    print("Checkpoint da CNN já existe. Reutilizando:")
    print(CNN_CHECKPOINT)
else:
    print("Checkpoint não encontrado.")
    print("Executando Feature Extraction + Fine-Tuning...")

    subprocess.run(
        [sys.executable, "-m", "src.cnn.train"],
        cwd=PROJECT_ROOT,
        check=True,
    )

## 10. Curvas de treinamento

As curvas permitem observar:

- redução da loss;
- evolução da accuracy;
- diferença entre treino e validação;
- efeito da transição de feature extraction para fine-tuning.

No experimento original:

### Feature Extraction

```text
Epoch 1:
Train Accuracy ≈ 84,85%
Val Accuracy   ≈ 98,11%

Epoch 2:
Train Accuracy ≈ 97,65%
Val Accuracy   ≈ 98,87%
```

Depois o backbone foi descongelado para fine-tuning.

O melhor checkpoint ocorreu na **época 4 da fase de Fine-Tuning**, com:

```text
Validation Loss     = 0,0202
Validation Accuracy = 99,25%
```

In [ ]:
import json
import pandas as pd
from IPython.display import display, Image as IPyImage

HISTORY_PATH = (
    PROJECT_ROOT
    / "results"
    / "cnn_pretrained_history.json"
)

with open(
    HISTORY_PATH,
    "r",
    encoding="utf-8",
) as arquivo:
    history = json.load(arquivo)

history_df = pd.DataFrame({
    "fase": history["fase"],
    "train_loss": history["train_loss"],
    "val_loss": history["val_loss"],
    "train_accuracy_%": [
        valor * 100
        for valor in history["train_accuracy"]
    ],
    "val_accuracy_%": [
        valor * 100
        for valor in history["val_accuracy"]
    ],
    "tempo_s": history["epoch_time_seconds"],
})

history_df

In [ ]:
for nome in [
    "cnn_pretrained_loss.png",
    "cnn_pretrained_accuracy.png",
]:
    caminho = PROJECT_ROOT / "results" / nome

    if caminho.exists():
        print(nome)
        display(IPyImage(filename=str(caminho)))

## 11. Avaliação no conjunto de teste

O teste permanece separado durante treinamento e seleção do checkpoint.

Serão reportados:

- accuracy global;
- precision / recall / F1 weighted;
- precision / recall / F1 macro;
- accuracy de cada classe;
- matriz de confusão.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "src.cnn.evaluate"],
    cwd=PROJECT_ROOT,
    check=True,
)

In [ ]:
TEST_RESULTS_PATH = (
    PROJECT_ROOT
    / "results"
    / "cnn_pretrained_test_results.json"
)

with open(
    TEST_RESULTS_PATH,
    "r",
    encoding="utf-8",
) as arquivo:
    resultados = json.load(arquivo)

resumo_global = pd.DataFrame({
    "Métrica": [
        "Accuracy",
        "Precision weighted",
        "Recall weighted",
        "F1 weighted",
        "Precision macro",
        "Recall macro",
        "F1 macro",
    ],
    "Valor (%)": [
        resultados["test"]["accuracy"] * 100,
        resultados["test"]["precision_weighted"] * 100,
        resultados["test"]["recall_weighted"] * 100,
        resultados["test"]["f1_weighted"] * 100,
        resultados["test"]["precision_macro"] * 100,
        resultados["test"]["recall_macro"] * 100,
        resultados["test"]["f1_macro"] * 100,
    ],
})

resumo_global

In [ ]:
por_classe = pd.DataFrame([
    {
        "Classe": classe,
        "Accuracy (%)": dados["accuracy"] * 100,
        "Corretos": dados["corretos"],
        "Total": dados["total"],
    }
    for classe, dados in resultados["accuracy_por_classe"].items()
])

por_classe

In [ ]:
matriz_path = (
    PROJECT_ROOT
    / "results"
    / "cnn_pretrained_confusion_matrix.png"
)

if matriz_path.exists():
    display(
        IPyImage(
            filename=str(matriz_path)
        )
    )

## 12. Resultado validado

O modelo obteve:

| Métrica | Resultado |
|---|---:|
| Test Accuracy | **98,49%** |
| Precision weighted | **98,51%** |
| Recall weighted | **98,49%** |
| F1 weighted | **98,50%** |
| Macro F1 | **98,48%** |

### Accuracy por classe

| Classe | Resultado |
|---|---:|
| bike | 98,15% (53/54) |
| cars | 98,41% (62/63) |
| cats | 96,67% (29/30) |
| dogs | 96,77% (30/31) |
| flowers | 100% (32/32) |
| horses | 100% (28/28) |
| human | 100% (27/27) |

Foram classificadas corretamente **261 das 265 imagens de teste**.

Os resultados foram reproduzidos novamente depois da reorganização do projeto, confirmando que os checkpoints e o pipeline continuaram consistentes.

## 13. Análise das estratégias de augmentation

A atividade pede não apenas aplicar augmentation, mas discutir **por que cada transformação faz sentido e onde pode prejudicar**.

| Estratégia | Benefício | Possível risco |
|---|---|---|
| Horizontal flip | variação de orientação esquerda/direita sem mudar a classe | inadequado se lateralidade ou texto importarem |
| Rotação leve | robustez a pequenos ângulos de câmera | rotação intensa criaria posições artificiais, especialmente para veículos/pessoas |
| Random resized crop | robustez a enquadramento, posição e escala | crop forte pode cortar partes essenciais do objeto |
| Color jitter | robustez a iluminação/câmera | alteração forte pode destruir pistas de cor, especialmente em flores |

### Outras estratégias que poderiam ser testadas

**Random Grayscale:** poderia reduzir dependência excessiva de cor, mas deve ser avaliado com cuidado porque `flowers` pode utilizar cor como informação útil.

**Multi-scale training:** poderia aumentar robustez a diferentes tamanhos dos objetos. Em contrapartida, aumenta custo computacional e exige cuidado com dimensões e batching.

**Augmentation específica por classe:** seria possível adaptar intensidades para categorias específicas, evitando transformações que removam informações relevantes.

## 14. Por que o resultado ficou tão alto?

Um resultado de 98,49% deve ser analisado criticamente.

### Fatores favoráveis

1. as classes são visualmente bastante distintas;
2. EfficientNet-B0 já possui conhecimento visual do ImageNet;
3. várias categorias têm correspondência conceitual próxima às classes do pré-treinamento;
4. o dataset foi deduplicado antes do split;
5. augmentation ajuda a reduzir dependência de pequenas variações;
6. fine-tuning adapta as features ao novo conjunto.

### Cuidado com data leakage

Antes da deduplicação foram encontrados arquivos exatos repetidos. Se eles fossem separados aleatoriamente entre treino e teste, a avaliação poderia ser artificialmente inflada.

Por isso a ordem correta foi:

```text
dataset original
 ↓
limpeza
 ↓
deduplicação
 ↓
split
 ↓
treinamento
```

## 15. Feature Extraction ou Fine-Tuning?

A escolha depende principalmente de:

- tamanho do novo dataset;
- similaridade com o domínio de pré-treinamento;
- capacidade computacional;
- risco de overfitting.

### Preferir Feature Extraction quando

```text
dataset pequeno
+
domínio parecido com ImageNet
+
pouca computação
```

### Considerar Fine-Tuning quando

```text
precisamos adaptar features
+
temos validação confiável
+
usamos learning rate menor
```

No EcoVision AI usamos as duas estratégias sequencialmente.

O dataset é pequeno, então começamos de forma conservadora. Depois, o backbone foi liberado com learning rate menor para adaptação fina.

## 16. Conceitos para estudo

### Convolução

Uma CNN aplica filtros que percorrem regiões locais da imagem.

Nas primeiras camadas, é comum aprender padrões simples como:

```text
bordas
texturas
contrastes
```

Camadas posteriores combinam esses padrões em representações mais abstratas.

### Backbone

É a parte principal da rede responsável por extrair features.

```text
imagem
 ↓
BACKBONE
 ↓
vetor de features
 ↓
classification head
 ↓
classe
```

### Classification Head

É a parte final que transforma as features em logits das classes da tarefa.

### Logits

São scores não normalizados produzidos pelo modelo.

### Backpropagation

```text
forward
 ↓
logits
 ↓
loss
 ↓
backward()
 ↓
gradientes
 ↓
optimizer.step()
 ↓
pesos atualizados
```

Durante feature extraction, gradientes alteram apenas o head.  
Durante fine-tuning, eles também alteram o backbone.

## 17. Conclusão

A EfficientNet-B0 pré-treinada mostrou forte capacidade de adaptação ao novo domínio.

O fluxo experimental foi:

```text
Kaggle
 ↓
1.803 imagens válidas
 ↓
deduplicação SHA-256
 ↓
1.764 imagens únicas
 ↓
split estratificado 70/15/15
 ↓
data augmentation
 ↓
EfficientNet-B0 ImageNet
 ↓
troca do head 1000 → 7
 ↓
Feature Extraction
 ↓
Fine-Tuning
 ↓
teste independente
 ↓
98,49% accuracy
```

O alto desempenho não elimina a necessidade de análise metodológica. A deduplicação, separação correta do teste e avaliação por classe são essenciais para que a métrica seja interpretável.

## 18. Checklist da rubrica — Atividade 3

- [x] CNN pré-treinada carregada
- [x] EfficientNet-B0 justificada para o domínio e Colab T4
- [x] Classification head substituído de 1000 para 7 classes
- [x] Backbone congelado
- [x] Feature extraction executado
- [x] Fine-tuning executado como etapa adicional
- [x] Curvas de loss por epoch
- [x] Curvas de accuracy por epoch
- [x] Accuracy global reportada
- [x] Accuracy por classe reportada
- [x] Pelo menos 3 augmentations — foram utilizadas 4
- [x] Justificativa específica das augmentations
- [x] Discussão de possíveis distorções por classe
- [x] Normalização compatível com ImageNet
- [x] Discussão Feature Extraction × Fine-Tuning
- [x] Dataset deduplicado antes do split
- [x] Verificação de data leakage entre splits
- [x] Matriz de confusão
- [x] Análise crítica dos resultados